# Module 2 Assignment — Computational Lab 2
Hello professor, the disease I have chosen is Ovarian carcinoma as it's one of the most common gynecological cancers. 
## Data Sources
| Layer | Source | Dataset | Access |
|-------|--------|---------|--------|
| Transcriptomics (RNA) | CPTAC-OV via LinkedOmics | RNAseq HiSeq, Gene level, log2 FPKM (82 tumor, no matched normal) | Open |
| Proteomics (protein) | CPTAC-OV via LinkedOmics | Proteome TMT, Gene level, Tumor (83) vs Normal (20) | Open |
| Genomics (GWAS) | GWAS Catalog (EBI) | Ovarian carcinoma (MONDO_0005140, with child traits) | Open |

RNA and protein are from the same CPTAC-OV cohort so they are matched but GWAS data is from a different population study. Everything has been integrated at the gene level

Additionally, there was no normal RNA data available so I couldn't do tumor vs normal comparison for RNA. I have used deviation from cohort mean instead of real tumor vs normal log2FC for RNA, which I thought was the best solution.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

## Helper function for scoring
I have given RNA less weight as there are no normal samples, the protein and GWAS data is more reliable

In [2]:
CUSTOM_WEIGHTS = {"transcriptomic": 0.20, "proteomic": 0.40, "genomic": 0.40}

def multi_evidence_score(df, layer_cols, weights):
    scored = df.copy()
    for col in layer_cols:
        mn, mx = scored[col].min(), scored[col].max()
        scored[col + "_norm"] = (scored[col] - mn) / (mx - mn) if mx > mn else 0.0
    scored["score"] = sum(
        scored[col + "_norm"] * weights[col] for col in layer_cols
    )
    return scored

## 3.1-Loading the data
Loading all three layers. RNA only has tumor samples so thats going to be handled differently than protein which has both tumor and normal.

In [3]:
# loading the CPTAC-OV files I downloaded from linkedomics
rna = pd.read_csv(r"C:\Users\zaina\biot6900\Module 2\Data\HS_CPTAC_OV_rnaseq_fpkm_log2.cct", sep="\t", index_col=0)
prot_tumor = pd.read_csv(r"C:\Users\zaina\biot6900\Module 2\Data\HS_CPTAC_OV_proteome_gene_tumor.cct", sep="\t", index_col=0)
prot_normal = pd.read_csv(r"C:\Users\zaina\biot6900\Module 2\Data\HS_CPTAC_OV_proteome_gene_normal.cct", sep="\t", index_col=0)

print(f"RNA: {rna.shape[0]} genes x {rna.shape[1]} samples (tumor only, no normal)")
print(f"Protein tumor: {prot_tumor.shape[0]} genes x {prot_tumor.shape[1]} samples")
print(f"Protein normal: {prot_normal.shape[0]} genes x {prot_normal.shape[1]} samples")

RNA: 22014 genes x 82 samples (tumor only, no normal)
Protein tumor: 10078 genes x 83 samples
Protein normal: 10078 genes x 20 samples


### RNA processing
Since there's no normal RNA tissue I can't do a real tumor vs normal comparison. Instead im computing how far each gene's expression deviates from the average across all genes, and using a z-score to get a p-value.

In [4]:
# computing RNA "effect" as deviation from global mean
# since theres no normal samples to compare against
rna_results = []
for gene in rna.index:
    vals = rna.loc[gene].dropna().values.astype(float)
    if len(vals) < 2 or vals.std() == 0:
        continue
    rna_results.append({"gene": gene, "mean_expr": vals.mean(), "std_expr": vals.std()})

rna_df = pd.DataFrame(rna_results)

# z-score each gene relative to the global mean
overall_mean = rna_df["mean_expr"].mean()
overall_std = rna_df["mean_expr"].std()
rna_df["log2fc"] = rna_df["mean_expr"] - overall_mean
rna_df["zscore"] = (rna_df["mean_expr"] - overall_mean) / overall_std
rna_df["pval"] = 2 * norm.sf(rna_df["zscore"].abs())

tx = rna_df[["gene", "log2fc", "pval"]].copy()
print(f"RNA genes after processing: {len(tx)}")
tx.sort_values("log2fc", key=abs, ascending=False).head()

RNA genes after processing: 19325


,gene,log2fc,pval
12624,B2M,9.236195,7.368306e-07
0,RPL30,9.218912,7.727348e-07
1,TMSB10,9.040087,1.258077e-06
15009,RPS6,8.988703,1.444830e-06
15685,EEF1A1,8.841693,2.138147e-06


### Protein differential expression
This is done using Welch's t-test like we did in assignment 1.

In [5]:
# protein tumor vs normal — this is the real DE comparison
prot_results = []
for gene in prot_tumor.index:
    t_vals = prot_tumor.loc[gene].dropna().values.astype(float)
    n_vals = prot_normal.loc[gene].dropna().values.astype(float) if gene in prot_normal.index else np.array([])
    if len(t_vals) < 2 or len(n_vals) < 2:
        continue
    log2fc = t_vals.mean() - n_vals.mean()
    _, pval = stats.ttest_ind(t_vals, n_vals, equal_var=False)
    prot_results.append({"gene": gene, "log2fc": log2fc, "pval": pval})

pr = pd.DataFrame(prot_results)
print(f"Protein DE genes: {len(pr)}")
pr.head()

Protein DE genes: 10056


,gene,log2fc,pval
0,A2M,-1.707575,9.991133e-09
1,ACADM,-0.095384,5.460648e-01
2,ACADS,-0.615442,5.260506e-03
3,ACADVL,-0.192308,1.369088e-01
4,ACAT1,-0.509969,1.467451e-03


### GWAS data
Downloaded from the GWAS Catalog — I searched "ovarian carcinoma" and picked the main trait (MONDO_0005140) with child traits included so it captures all subtypes like serous, endometrioid etc.

In [6]:
# loading gwas catalog download
gw_raw = pd.read_csv(r"C:\Users\zaina\biot6900\Module 2\Data\gwas-association-downloaded_2026-09-24-MONDO_0005140-withChildTraits.tsv", sep="\t")
print(f"Raw GWAS rows: {len(gw_raw)}")

# some rows have multiple genes separated by dashes or commas, need to split those
gw = gw_raw.copy()
gw["gene"] = gw["MAPPED_GENE"].astype(str).str.split(r"[\s]*[-,;]\s*")
gw = gw.explode("gene")
gw = gw[gw["gene"].str.strip().astype(bool)].copy()
gw["gene"] = gw["gene"].str.strip()

# convert p-value to -log10
gw["pvalue"] = pd.to_numeric(gw["P-VALUE"], errors="coerce")
gw = gw[gw["pvalue"] > 0].copy()
gw["neglog10p"] = -np.log10(gw["pvalue"])

# if a gene shows up multiple times keep the strongest signal
gw = gw.sort_values("neglog10p", ascending=False).drop_duplicates(subset="gene", keep="first")
gw = gw[["gene", "neglog10p"]].reset_index(drop=True)
print(f"GWAS genes after cleanup: {len(gw)}")
gw.sort_values("neglog10p", ascending=False).head()

Raw GWAS rows: 732
GWAS genes after cleanup: 557


,gene,neglog10p
0,FGFR2,253.39794
1,ABO,88.00000
2,NaN,70.09691
3,RN7SL720P,63.30103
4,BNC2,63.30103


## 3.2 — Joining everything on gene symbol
Renaming columns first so RNA and protein dont clash when I merge them, then doing a 3-way inner join on the gene name.

In [7]:
# rename so columns dont collide
tx = tx.rename(columns={"log2fc": "rna_lfc", "pval": "rna_p"})
pr = pr.rename(columns={"log2fc": "prot_lfc", "pval": "prot_p"})

# merge all three on gene
df = tx.merge(pr, on="gene").merge(gw, on="gene")
print(f"Genes surviving the 3-way join: {len(df)}")
# lost a lot of genes here because GWAS only covers ~557 genes

Genes surviving the 3-way join: 154


## 3.3 — Concordance
Checking if RNA and protein agree on direction (both up or both down). Since RNA here isn't a true tumor vs normal comparison the concordance is a bit weaker than it would be with proper normal samples, but still worth looking at.

In [8]:
df["concordant"] = np.sign(df["rna_lfc"]) == np.sign(df["prot_lfc"])
print(f"Concordant genes: {df['concordant'].sum()} / {len(df)}")
print(f"Concordance rate: {df['concordant'].mean():.1%}")

Concordant genes: 92 / 154
Concordance rate: 59.7%


## 3.4 — Multi-evidence score
Using custom weights here — RNA gets 0.20 since it doesn't have a real tumor vs normal baseline, and protein (0.40) and GWAS (0.40) get more because they're more reliable layers.

In [9]:
df["transcriptomic"] = df["rna_lfc"].abs()
df["proteomic"]      = df["prot_lfc"].abs()
df["genomic"]        = df["neglog10p"]

df = multi_evidence_score(df, ["transcriptomic", "proteomic", "genomic"], CUSTOM_WEIGHTS)

print(f"Score range: {df['score'].min():.3f} - {df['score'].max():.3f}")
df.sort_values("score", ascending=False).head(10)

Score range: 0.003 - 0.530


,gene,rna_lfc,rna_p,prot_lfc,prot_p,neglog10p,concordant,transcriptomic,proteomic,genomic,transcriptomic_norm,proteomic_norm,genomic_norm,score
22,FGFR2,1.868571,0.316481,-0.425025,5.357095e-02,253.397940,False,1.868571,0.425025,253.397940,0.304481,0.172938,1.000000,0.530071
92,HSPB7,-1.244562,0.504649,-2.224439,1.223323e-06,8.154902,True,1.244562,2.224439,8.154902,0.202260,0.916672,0.012519,0.412128
67,SYNPO2,-0.280274,0.880567,-2.426046,3.499533e-09,5.522879,True,0.280274,2.426046,5.522879,0.044297,1.000000,0.001921,0.409628
88,SGCG,-1.078651,0.563096,-1.990384,4.561641e-03,5.154902,True,1.078651,1.990384,5.154902,0.175082,0.819932,0.000439,0.363165
72,DMD,-0.346411,0.852675,-1.933062,1.553644e-09,6.397940,True,0.346411,1.933062,6.397940,0.055131,0.796240,0.005445,0.331700
113,CD248,-0.627428,0.736603,-1.779659,2.048286e-13,8.221849,True,0.627428,1.779659,8.221849,0.101165,0.732835,0.012789,0.318483
94,THSD7A,-1.329723,0.475941,-1.596009,1.149023e-09,7.000000,True,1.329723,1.596009,7.000000,0.216210,0.656929,0.007869,0.309161
98,VIT,-1.596525,0.392067,-1.441845,5.178903e-02,5.096910,True,1.596525,1.441845,5.096910,0.259916,0.593210,0.000206,0.289350
1,TUBA1C,6.114384,0.001046,0.338427,3.006823e-03,6.045757,True,6.114384,0.338427,6.045757,1.000000,0.137146,0.004027,0.256469
124,CCDC170,-0.547410,0.769172,-1.404420,6.474857e-02,5.397940,True,0.547410,1.404420,5.397940,0.088057,0.577742,0.001418,0.249275


## 3.5 — Ranking and export

In [10]:
ranked = df.sort_values("score", ascending=False)
top = ranked.head(15)
ranked.to_csv("targets_ovarian.csv", index=False)

print(top[["gene", "rna_lfc", "prot_lfc", "neglog10p", "concordant", "score"]].round(3).to_string(index=False))

   gene  rna_lfc  prot_lfc  neglog10p  concordant  score
  FGFR2    1.869    -0.425    253.398       False  0.530
  HSPB7   -1.245    -2.224      8.155        True  0.412
 SYNPO2   -0.280    -2.426      5.523        True  0.410
   SGCG   -1.079    -1.990      5.155        True  0.363
    DMD   -0.346    -1.933      6.398        True  0.332
  CD248   -0.627    -1.780      8.222        True  0.318
 THSD7A   -1.330    -1.596      7.000        True  0.309
    VIT   -1.597    -1.442      5.097        True  0.289
 TUBA1C    6.114     0.338      6.046        True  0.256
CCDC170   -0.547    -1.404      5.398        True  0.249
   CD44    1.894    -1.120      5.046       False  0.246
   RALY    4.391     0.565      7.699        True  0.240
   LMNA    3.814    -0.645      5.699       False  0.231
  MECOM    5.832     0.231      6.699        True  0.231
  DCLK1   -1.225    -1.131      5.222        True  0.226


### Sanity check — do known ovarian cancer genes show up?
Checking if well-known ovarian cancer genes made it into the results. A lot of the classic ones like BRCA1/2 probably wont be here since GWAS captures inherited risk not somatic mutations, and the inner join drops anything that doesn't appear in all three layers.

In [11]:
# checking for known ovarian cancer genes
known_ov = ["BRCA1", "BRCA2", "TP53", "PAX8", "FGFR2", "MECOM", "ESR1", "CD44", "LMNA", "MUC16", "WFDC2"]

gene_list = list(ranked["gene"])
for g in known_ov:
    row = ranked[ranked["gene"] == g]
    if len(row) > 0:
        rank_pos = gene_list.index(g) + 1
        s = row.iloc[0]
        print(f"  {g:10s}  rank={rank_pos:>4d}/{len(ranked)}  score={s['score']:.3f}  "
              f"rna={s['rna_lfc']:+.3f}  prot={s['prot_lfc']:+.3f}  concordant={s['concordant']}")
    else:
        print(f"  {g:10s}  - not in joined table")

  BRCA1       - not in joined table
  BRCA2       - not in joined table
  TP53        rank=  66/154  score=0.109  rna=+1.825  prot=+0.268  concordant=True
  PAX8        rank=  25/154  score=0.184  rna=+3.541  prot=+0.357  concordant=True
  FGFR2       rank=   1/154  score=0.530  rna=+1.869  prot=-0.425  concordant=False
  MECOM       rank=  14/154  score=0.231  rna=+5.832  prot=+0.231  concordant=True
  ESR1        rank=  35/154  score=0.156  rna=+2.049  prot=-0.410  concordant=False
  CD44        rank=  11/154  score=0.246  rna=+1.894  prot=-1.120  concordant=False
  LMNA        rank=  13/154  score=0.231  rna=+3.814  prot=-0.645  concordant=False
  MUC16       - not in joined table
  WFDC2       - not in joined table


### Discordant genes
Looking at genes where RNA and protein disagree — these are interesting to write about in the report because you have to think about why they might not match up.

In [12]:
discordant_top = ranked.head(50).query("concordant == False")
print(f"Discordant genes in top 50: {len(discordant_top)}")
if len(discordant_top) > 0:
    print(discordant_top[["gene", "rna_lfc", "prot_lfc", "neglog10p", "score"]].round(3).to_string(index=False))

Discordant genes in top 50: 13
  gene  rna_lfc  prot_lfc  neglog10p  score
 FGFR2    1.869    -0.425    253.398  0.530
  CD44    1.894    -1.120      5.046  0.246
  LMNA    3.814    -0.645      5.699  0.231
 MXRA8    1.599    -0.793      9.301  0.189
SORBS2    0.186    -1.052      5.699  0.180
NDUFS4    4.239    -0.093      8.155  0.158
 INHBB    0.861    -0.768      7.523  0.158
  ESR1    2.049    -0.410     19.046  0.156
  GPAM    0.199    -0.903      5.155  0.155
ATP2B4    1.961    -0.527      5.523  0.151
ACTR10    3.071    -0.261      5.523  0.143
  GCLC    1.280    -0.560      8.155  0.138
 LAMA3    1.372    -0.546      7.398  0.138
